<a href="https://colab.research.google.com/github/shreebiyani/Fake-News-Detection-using-NLP/blob/main/C2_21_37_NLP_Poject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Statements**


In [ ]:
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


**Loading Dataset**

In [ ]:
print("\n[1] Loading Dataset...")
url = "https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv"
df = pd.read_csv(url)
df = df[['text', 'label']]
df['label'] = df['label'].map({'FAKE': 0, 'REAL': 1})
print("Dataset Loaded Successfully!")
print("Total Samples:", df.shape[0])
print("Fake News:", (df['label'] == 0).sum())
print("Real News:", (df['label'] == 1).sum())



[1] Loading Dataset...
Dataset Loaded Successfully!
Total Samples: 6335
Fake News: 3164
Real News: 3171


**Text Preprocessing**

In [ ]:
print("\n[2] Preprocessing Text Data...")

nltk.download('stopwords')
stop_words = stopwords.words('english')

def preprocess(text):
    text = text.lower()
    text = ''.join([c for c in text if c not in string.punctuation])
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

print("Sample Before Cleaning:")
print(df['text'].iloc[0][:200])

print("\nSample After Cleaning:")
print(df['clean_text'].iloc[0][:200])



[2] Preprocessing Text Data...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Sample Before Cleaning:
Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. 
In the final stretch of the election, Hillary Rodham Clinton has gone to war wit

Sample After Cleaning:
daniel greenfield shillman journalism fellow freedom center new york writer focusing radical islam final stretch election hillary rodham clinton gone war fbi word “unprecedented” thrown around often e


**Feature Extraction**

In [ ]:
print("\n[3] Extracting Features using TF-IDF...")

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

print("Feature Matrix Shape:", X.shape)




[3] Extracting Features using TF-IDF...
Feature Matrix Shape: (6335, 5000)


**Model Training**

In [ ]:
print("\n[4] Training Logistic Regression Model...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model Training Completed!")



[4] Training Logistic Regression Model...
Model Training Completed!


**Model Evaluation**

In [ ]:
print("\n[5] Evaluating Model Performance...")

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



[5] Evaluating Model Performance...
Accuracy: 0.9218626677190213

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.93      0.92       628
           1       0.93      0.91      0.92       639

    accuracy                           0.92      1267
   macro avg       0.92      0.92      0.92      1267
weighted avg       0.92      0.92      0.92      1267



**Real-Time Prediction**

In [ ]:
print("\n[6] Testing with New News Article...")

def predict_news(text):
    text = preprocess(text)
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)
    return "REAL NEWS" if prediction[0] == 1 else "FAKE NEWS"

sample_news = "Breaking news: scientists discover a new planet"
print("Input News:", sample_news)
print("Prediction:", predict_news(sample_news))



[6] Testing with New News Article...
Input News: Breaking news: scientists discover a new planet
Prediction: FAKE NEWS


In [ ]:
import pickle

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Save the trained model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model and vectorizer saved!")

Model and vectorizer saved!
